# Pet Image Classifier (Cat / Dog / Parrot)

**Dataset folder structure (required):**
```
dataset/
    cat/      -> cat images
    dog/      -> dog images
    parrot/   -> parrot images
```
Folder names = class names. Add more folders (e.g. `rabbit/`) and it will learn them automatically.

**Google Colab:** zip your `dataset` folder as `dataset.zip` and upload it in Step 2.

## Step 1: Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

print('TensorFlow version:', tf.__version__)

## Step 2: Load your dataset
Run the Colab upload cell only if you are on Google Colab. Skip it if the `dataset` folder is already next to this notebook.

In [ ]:
# --- Google Colab only: upload dataset.zip and unzip it ---
try:
    from google.colab import files
    uploaded = files.upload()          # choose dataset.zip
    !unzip -q -o dataset.zip
except ImportError:
    print('Not on Colab - using local folder.')

In [ ]:
DATA_DIR = 'dataset'      # change if your folder has a different name
IMG_SIZE = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH)

class_names = train_ds.class_names
print('Classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## Step 3: Look at some images

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.show()

## Step 4: Build the model (transfer learning)
We reuse MobileNetV2 (already trained on millions of images) and only train a small new layer on top. This is fast and accurate.

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    augment,
    layers.Rescaling(1./127.5, offset=-1),   # MobileNetV2 expects -1..1
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## Step 5: Train

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Accuracy'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Loss'); plt.legend()
plt.show()

## Step 6: Test with your own pet photo
Run the cell, upload a photo, and see which pet it is.

In [ ]:
def predict_pet(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[None, ...]
    probs = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))
    plt.imshow(img); plt.axis('off')
    plt.title(f'{class_names[idx]}  ({probs[idx]*100:.1f}%)')
    plt.show()
    for name, p in zip(class_names, probs):
        print(f'{name}: {p*100:.1f}%')

# Colab: upload an image
try:
    from google.colab import files
    up = files.upload()
    for fname in up.keys():
        predict_pet(fname)
except ImportError:
    predict_pet('my_pet.jpg')   # local: put your image path here

## Step 7 (optional): Save the model

In [ ]:
model.save('pet_classifier.keras')
# Later: model = tf.keras.models.load_model('pet_classifier.keras')